# Etiquetado de Componentes Conexos

In [1]:
%export QT_DEBUG_PLUGINS=1

UsageError: Line magic function `%export` not found.


In [ ]:
import cv2
import numpy as np

In [ ]:
# Leer imagen en gris
imagen = cv2.imread("img/IMG_20191209_100620.jpg", cv2.IMREAD_GRAYSCALE)
imagen_color = cv2.cvtColor(imagen, cv2.COLOR_GRAY2BGR)

# =========================
# 1. BINARIZACIÓN
# =========================

_, imagen_bin = cv2.threshold(imagen, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

# =========================
# 2. LIMPIEZA
# =========================
kernel = np.ones((3,3), np.uint8)
imagen_bin = cv2.morphologyEx(imagen_bin, cv2.MORPH_CLOSE, kernel)
imagen_bin = cv2.morphologyEx(imagen_bin, cv2.MORPH_OPEN, kernel)


cv2.imshow("Bin", imagen_bin)
cv2.waitKey(0)


-1

: 

: 

In [ ]:

# Etiquetar componentes (dados)
num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(imagen_bin, 8, cv2.CV_32S)


: 

: 

In [ ]:

contador_dados = 0

for i in range(1, num_labels):
    x = stats[i, cv2.CC_STAT_LEFT]
    y = stats[i, cv2.CC_STAT_TOP]
    w = stats[i, cv2.CC_STAT_WIDTH]
    h = stats[i, cv2.CC_STAT_HEIGHT]
    area = stats[i, cv2.CC_STAT_AREA]

    # Filtrar dados (grandes)
    if 3000 < area < 30000:
        contador_dados += 1

        roi = imagen[y:y+h, x:x+w]

        # =========================
        # 4. DETECTAR PUNTOS
        # =========================

        # binarizar ROI
        _, roi_bin = cv2.threshold(roi, 120, 255, cv2.THRESH_BINARY_INV)
        #roi_bin = cv2.morphologyEx(roi_bin, cv2.MORPH_OPEN, np.ones((3,3), np.uint8))
        #cv2.imshow("ROI", roi)
        #cv2.imshow("ROI binaria", roi_bin)
        #cv2.waitKey(0)
        # componentes en ROI
        n_labels, _, stats_p, _ = cv2.connectedComponentsWithStats(roi_bin, 8)

        puntos = 0

        for j in range(1, n_labels):
            area_p = stats_p[j, cv2.CC_STAT_AREA]

            # filtrar puntos (pequeños)
            if 1 < area_p < 100:
                puntos += 1

        # Dibujar
        cv2.rectangle(imagen_color, (x,y), (x+w,y+h), (0,255,0), 2)
        cv2.putText(imagen_color, str(puntos), (x, y-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,0), 2)


print("Número de dados:", contador_dados)

# =========================
# MOSTRAR
# =========================
bin_color = cv2.cvtColor(imagen_bin, cv2.COLOR_GRAY2BGR)
resultado = np.hstack((imagen_color, bin_color))

cv2.namedWindow("Resultado", cv2.WINDOW_NORMAL)
cv2.imshow("Resultado", imagen_color)
cv2.waitKey(0)
cv2.destroyAllWindows()

Número de dados: 4


: 

: 

## De la documentación de OpenCV
### connectedComponentsWithStats
#### Parameters
- image	the 8-bit single-channel image to be labeled
- labels	destination labeled image
- stats	statistics output for each label, including the background label. Statistics are accessed via stats(label, COLUMN) where COLUMN is one of ConnectedComponentsTypes, selecting the statistic. The data type is CV_32S.
- centroids	centroid output for each label, including the background label. Centroids are accessed via centroids(label, 0) for x and centroids(label, 1) for y. The data type CV_64F.
- connectivity	8 or 4 for 8-way or 4-way connectivity respectively
- ltype	output image label type. Currently CV_32S and CV_16U are supported.

---

Los 'stats' están definidos por los siguientes tipos:
- CC_STAT_LEFT 
Python: cv.CC_STAT_LEFT
The leftmost (x) coordinate which is the inclusive start of the bounding box in the horizontal direction.
- CC_STAT_TOP 
Python: cv.CC_STAT_TOP
The topmost (y) coordinate which is the inclusive start of the bounding box in the vertical direction.
- CC_STAT_WIDTH 
Python: cv.CC_STAT_WIDTH
The horizontal size of the bounding box.
- CC_STAT_HEIGHT 
Python: cv.CC_STAT_HEIGHT
The vertical size of the bounding box.
- CC_STAT_AREA 
Python: cv.CC_STAT_AREA
The total area (in pixels) of the connected component.